# Synthetic validation — summary tables

Hardcoded metrics aligned with the Streamlit re-run (±**10 mm** reference window).

**Exports:** `table_b_synthetic_1_validation.csv`, `table_c_synthetic_2_validation.csv`,
`table_e_colormap_ranges.csv`.

## 1. Setup

In [1]:
from __future__ import annotations

import math
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "src" / "blocks" / "_01_extraction.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(
            "Run from notebooks/results visualizations/synthetic."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPORT_DIR = NOTEBOOK_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_WINDOW_MM = 10.0
THEO_AREA_UNIFORM_MM2 = math.pi * 10.0**2

# Single source of truth for the `case` column (identical in tables B, C, and E).
CASE_LABELS: dict[str, str] = {
    "Synthetic_1": "Healthy control (uniform tube)",
    "Synthetic_2": "Stenosis control (cosine waist)",
}

print(f"Exports: {EXPORT_DIR}")
for pid, label in CASE_LABELS.items():
    print(f"  {pid}: {label}")

Exports: C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\synthetic\exports
  Synthetic_1: Healthy control (uniform tube)
  Synthetic_2: Stenosis control (cosine waist)


## 2. Hardcoded metrics

In [2]:
def _vis_row(patient_id: str, **metrics: float) -> dict:
    return {
        "patient_id": patient_id,
        "case": CASE_LABELS[patient_id],
        "reference_window_mm": REFERENCE_WINDOW_MM,
        **metrics,
    }


SYNTHETIC_1_VIS = _vis_row(
    "Synthetic_1",
    theoretical_max_pct_as_pct=0.00,
    theoretical_area_mm2=round(THEO_AREA_UNIFORM_MM2, 2),
    predicted_max_pct_as_pct=0.01,
    predicted_mean_area_mm2=316.50,
    abs_error_max_pct_as_pct=0.01,
    abs_error_mean_area_mm2=2.34,
    colormap_area_min_mm2=316.495,
    colormap_area_max_mm2=316.525,
    colormap_pctas_min_pct=0.00,
    colormap_pctas_max_pct=0.0088,
)

SYNTHETIC_2_VIS = _vis_row(
    "Synthetic_2",
    theoretical_max_pct_as_pct=75.00,
    theoretical_max_area_mm2=round(THEO_AREA_UNIFORM_MM2, 2),
    theoretical_min_area_mm2=78.54,
    predicted_max_pct_as_pct=73.69,
    predicted_max_area_mm2=318.66,
    predicted_min_area_mm2=82.55,
    abs_error_max_pct_as_pct=1.31,
    abs_error_max_area_mm2=4.50,
    abs_error_min_area_mm2=4.01,
    colormap_area_min_mm2=83.0,
    colormap_area_max_mm2=320.0,
    colormap_pctas_min_pct=0.00,
    colormap_pctas_max_pct=74.0,
)

## 3. Build tables

In [3]:
def _row_meta(vis: dict) -> dict:
    pid = vis["patient_id"]
    return {
        "patient_id": pid,
        "case": CASE_LABELS[pid],
        "reference_window_mm": vis["reference_window_mm"],
    }


def _validation_long_table(vis: dict, *, include_min_max_area: bool) -> pd.DataFrame:
    specs: list[tuple[str, str, float | None, float | None, str | None]] = [
        (
            "Maximum % area stenosis",
            "%",
            vis["theoretical_max_pct_as_pct"],
            vis["predicted_max_pct_as_pct"],
            "abs_error_max_pct_as_pct",
        ),
    ]
    if include_min_max_area:
        specs.extend(
            [
                (
                    "Maximum cross-sectional area",
                    "mm²",
                    vis["theoretical_max_area_mm2"],
                    vis["predicted_max_area_mm2"],
                    "abs_error_max_area_mm2",
                ),
                (
                    "Minimum cross-sectional area",
                    "mm²",
                    vis["theoretical_min_area_mm2"],
                    vis["predicted_min_area_mm2"],
                    "abs_error_min_area_mm2",
                ),
            ]
        )
    else:
        specs.append(
            (
                "Mean cross-sectional area",
                "mm²",
                vis["theoretical_area_mm2"],
                vis["predicted_mean_area_mm2"],
                "abs_error_mean_area_mm2",
            )
        )

    rows: list[dict] = []
    for metric, unit, theo, pred, err_key in specs:
        err = (
            vis[err_key]
            if err_key
            else (
                abs(float(theo) - float(pred))
                if theo is not None and pred is not None
                else None
            )
        )
        rows.append(
            {
                **_row_meta(vis),
                "metric": metric,
                "unit": unit,
                "theoretical": theo,
                "predicted": pred,
                "absolute_error": err,
            }
        )
    return pd.DataFrame(rows)


def _colormap_table(vis: dict) -> pd.DataFrame:
    meta = _row_meta(vis)
    return pd.DataFrame(
        [
            {
                **meta,
                "colormap": "Area",
                "min": vis["colormap_area_min_mm2"],
                "max": vis["colormap_area_max_mm2"],
                "unit": "mm²",
            },
            {
                **meta,
                "colormap": "%AS",
                "min": vis["colormap_pctas_min_pct"],
                "max": vis["colormap_pctas_max_pct"],
                "unit": "%",
            },
        ]
    )


df_table_b = _validation_long_table(SYNTHETIC_1_VIS, include_min_max_area=False)
df_table_c = _validation_long_table(SYNTHETIC_2_VIS, include_min_max_area=True)
df_table_e = pd.concat(
    [_colormap_table(SYNTHETIC_1_VIS), _colormap_table(SYNTHETIC_2_VIS)],
    ignore_index=True,
)

TABLES = {
    "table_b_synthetic_1_validation": df_table_b,
    "table_c_synthetic_2_validation": df_table_c,
    "table_e_colormap_ranges": df_table_e,
}

for name, df in TABLES.items():
    for pid, expected in CASE_LABELS.items():
        subset = df.loc[df["patient_id"] == pid, "case"]
        if subset.empty:
            continue
        assert subset.nunique() == 1 and subset.iloc[0] == expected, (
            f"{name}: inconsistent case for {pid}"
        )

## 4. Preview

In [4]:
for name, df in TABLES.items():
    display(Markdown(f"### `{name}`"))
    display(df)

### `table_b_synthetic_1_validation`

,patient_id,case,reference_window_mm,metric,unit,theoretical,predicted,absolute_error
0,Synthetic_1,Healthy control (uniform tube),10.0,Maximum % area stenosis,%,0.00,0.01,0.01
1,Synthetic_1,Healthy control (uniform tube),10.0,Mean cross-sectional area,mm²,314.16,316.50,2.34


### `table_c_synthetic_2_validation`

,patient_id,case,reference_window_mm,metric,unit,theoretical,predicted,absolute_error
0,Synthetic_2,Stenosis control (cosine waist),10.0,Maximum % area stenosis,%,75.00,73.69,1.31
1,Synthetic_2,Stenosis control (cosine waist),10.0,Maximum cross-sectional area,mm²,314.16,318.66,4.50
2,Synthetic_2,Stenosis control (cosine waist),10.0,Minimum cross-sectional area,mm²,78.54,82.55,4.01


### `table_e_colormap_ranges`

,patient_id,case,reference_window_mm,colormap,min,max,unit
0,Synthetic_1,Healthy control (uniform tube),10.0,Area,316.495,316.5250,mm²
1,Synthetic_1,Healthy control (uniform tube),10.0,%AS,0.000,0.0088,%
2,Synthetic_2,Stenosis control (cosine waist),10.0,Area,83.000,320.0000,mm²
3,Synthetic_2,Stenosis control (cosine waist),10.0,%AS,0.000,74.0000,%


## 5. Export CSV

In [5]:
for name, df in TABLES.items():
    out = EXPORT_DIR / f"{name}.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"Wrote {out}")

Wrote C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\synthetic\exports\table_b_synthetic_1_validation.csv
Wrote C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\synthetic\exports\table_c_synthetic_2_validation.csv
Wrote C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\synthetic\exports\table_e_colormap_ranges.csv
